# Walk-Forward backtest notebook (BTC/ETH, 1D, 2020–2025)

Этот ноутбук — линейный сценарий, полностью заменяющий `runner.py`:

**загрузка данных → конфиг → генерация walk-forward окон → оптимизация на train → прогон на test с warm-up buffer → stitched OOS → отчёты → RC/SPA → robustness.**

> Предпосылка: в проекте лежит папка `src/` с модулями `wf.py`, `metrics.py`, `report.py`, `stats_tests.py` и стратегиями в `src/strategies/*`.


In [1]:
# ==== Setup (imports, logging, versions) ====
import json
import logging
from pathlib import Path
from typing import Any, Dict

import numpy as np
import pandas as pd

# plotting (matplotlib only; no seaborn)
import matplotlib.pyplot as plt

# --- Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("wf_notebook")

# --- Jupyter display helpers (explicit import so it works outside IPython too)
try:
    from IPython.display import display
except Exception:  # pragma: no cover
    display = print  # type: ignore

# --- Project root (assume notebook is run from repo root)
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(
        "Expected to run notebook from project root (folder containing 'src/'). "
        f"Current cwd={PROJECT_ROOT}"
    )
logger.info("PROJECT_ROOT = %s", PROJECT_ROOT)

# --- Core modules used early in the notebook
from src.wf import WFFold, generate_folds
from src.metrics import COST_SCENARIOS
from src import report as report_mod
from src import stats_tests as st

# --- Reproducibility hooks
np.random.seed(42)

# --- Versions for write-up reproducibility
import backtesting as _bt_pkg
logger.info(
    "pandas=%s | numpy=%s | backtesting.py=%s",
    pd.__version__, np.__version__, getattr(_bt_pkg, "__version__", "unknown")
)

2026-03-26 11:58:45,279 | INFO | PROJECT_ROOT = C:\Users\prosh\OneDrive\Рабочий стол\ДЗ МСК\4 course\Диплом\refactoring
C:\Users\prosh\.conda\envs\hse-ml\Lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

2026-03-26 11:58:46,667 | INFO | pandas=2.2.1 | numpy=1.23.5 | backtesting.py=0.6.5


In [2]:
from src.experiment_config import default_config, save_config_json
from src.utils.sizing_patch import apply_order_size_buffer

cfg = default_config(PROJECT_ROOT)

cfg.results_dir.mkdir(parents=True, exist_ok=True)
cfg.figures_dir.mkdir(parents=True, exist_ok=True)  # удобно иметь заранее
save_config_json(cfg, cfg.results_dir / "config_notebook.json")

apply_order_size_buffer(max_rel_size=float(cfg.bt.max_rel_size))

cfg

ExperimentConfig(data_paths={'BTCUSDT': WindowsPath('../data/data_raw/BTCUSDT_spot_1d_monthly_2020-01_2025-12.csv'), 'ETHUSDT': WindowsPath('../data/data_raw/ETHUSDT_spot_1d_monthly_2020-01_2025-12.csv')}, out_dir=WindowsPath('C:/Users/prosh/OneDrive/Рабочий стол/ДЗ МСК/4 course/Диплом/refactoring'), results_dir=WindowsPath('C:/Users/prosh/OneDrive/Рабочий стол/ДЗ МСК/4 course/Диплом/refactoring/results'), figures_dir=WindowsPath('C:/Users/prosh/OneDrive/Рабочий стол/ДЗ МСК/4 course/Диплом/refactoring/figures'), wf=WFConfig(train_years=3, test_months=6, step_months=6, warmup_bars=252, require_full_warmup=True), bt=BacktestConfig(cash=2000000.0, trade_on_close=False, exclusive_orders=True, hedging=False, margin=1.0, max_rel_size=0.999, finalize_trades=True), costs=['Low', 'Base', 'High'], objective=ObjectiveConfig(min_trades=10, min_exposure=0.1, penalty=1000000.0), rc_spa=RCSPAConfig(block_size=10, reps=2000, seed=42, studentize=True), artifacts=ArtifactsConfig(save_heatmaps=True, fail

## Загрузка и минимальная проверка данных

Требования:
- читаем CSV,
- индекс `DatetimeIndex`,
- сортировка,
- базовые sanity-checks по OHLC,
- приводим колонки в формат backtesting.py: `Open/High/Low/Close/Volume`.


In [3]:
from src.data_io import load_all_symbols

data_by_symbol = load_all_symbols(cfg.data_paths, project_root=PROJECT_ROOT, logger=logger)
{sym: df.head(3) for sym, df in data_by_symbol.items()}

2026-03-26 11:58:46,771 | INFO | BTCUSDT loaded: n=2192 | 2020-01-01 .. 2025-12-31
2026-03-26 11:58:46,803 | INFO | ETHUSDT loaded: n=2192 | 2020-01-01 .. 2025-12-31


{'BTCUSDT':                  Open    High      Low    Close        Volume
 datetime_utc                                                 
 2020-01-01    7195.24  7255.0  7175.15  7200.85  16792.388165
 2020-01-02    7200.77  7212.5  6924.74  6965.71  31951.483932
 2020-01-03    6965.49  7405.0  6871.04  7344.96  68428.500451,
 'ETHUSDT':                 Open    High     Low   Close        Volume
 datetime_utc                                              
 2020-01-01    129.16  133.05  128.68  130.77  144770.52197
 2020-01-02    130.72  130.78  126.38  127.19  213757.05806
 2020-01-03    127.19  135.14  125.88  134.35  413055.18895}

## Реестр стратегий (registry)

Структура реестра повторяет `runner.py`:
- `strategy_id` (код + имя),
- класс Strategy,
- grid параметров,
- constraint (если нужен),
- группа (trend/momentum/meanrev/synergy),
- сохраняем ли heatmap.

Buy&Hold добавляем как отдельный “bench-spec”, чтобы прогонять его тем же движком и с теми же издержками.


In [4]:
from src.strategy_registry import build_registry, get_benchmark, maybe_filter_debug

REGISTRY = build_registry()
BENCH = get_benchmark()

if cfg.debug.enabled:
    REGISTRY = maybe_filter_debug(REGISTRY, enabled=True, only_strategies=cfg.debug.only_strategies)

logger.info("Registry size (excluding bench): %d", len(REGISTRY))
[s.strategy_id for s in REGISTRY[:5]] + ["..."]

2026-03-26 11:58:46,833 | INFO | Registry size (excluding bench): 12


['T1:SMA_Crossover',
 'T2:EMA_Crossover',
 'T3:Donchian_Breakout',
 'M1:TSMOM',
 'R1:RSI_MR',
 '...']

## Генерация walk-forward окон

Для каждого актива строим окна через `wf.generate_folds()` и печатаем первые фолды (train/test/buffer_start), чтобы сразу увидеть, что warm-up действительно попадает в данные.


In [5]:
folds_by_symbol: dict[str, list[WFFold]] = {}

for sym, df in data_by_symbol.items():
    folds = generate_folds(
        df.index,
        train_years=cfg.wf.train_years,
        test_months=cfg.wf.test_months,
        step_months=cfg.wf.step_months,
        warmup_bars=cfg.wf.warmup_bars,
        require_full_warmup=cfg.wf.require_full_warmup,
    )
    folds_by_symbol[sym] = folds
    logger.info("%s folds: %d", sym, len(folds))

    for f in folds[:3]:
        print(
            f"[{sym}] fold {f.fold_id} | "
            f"train [{f.train_start.date()} .. {f.train_end.date()}) | "
            f"buffer_start={f.buffer_start.date()} | "
            f"test [{f.test_start.date()} .. {f.test_end.date()})"
        )

{k: len(v) for k, v in folds_by_symbol.items()}

2026-03-26 11:58:46,852 | INFO | BTCUSDT folds: 5
2026-03-26 11:58:46,857 | INFO | ETHUSDT folds: 5


[BTCUSDT] fold 0 | train [2020-01-01 .. 2023-01-01) | buffer_start=2022-04-24 | test [2023-01-01 .. 2023-07-01)
[BTCUSDT] fold 1 | train [2020-07-01 .. 2023-07-01) | buffer_start=2022-10-22 | test [2023-07-01 .. 2024-01-01)
[BTCUSDT] fold 2 | train [2021-01-01 .. 2024-01-01) | buffer_start=2023-04-24 | test [2024-01-01 .. 2024-07-01)
[ETHUSDT] fold 0 | train [2020-01-01 .. 2023-01-01) | buffer_start=2022-04-24 | test [2023-01-01 .. 2023-07-01)
[ETHUSDT] fold 1 | train [2020-07-01 .. 2023-07-01) | buffer_start=2022-10-22 | test [2023-07-01 .. 2024-01-01)
[ETHUSDT] fold 2 | train [2021-01-01 .. 2024-01-01) | buffer_start=2023-04-24 | test [2024-01-01 .. 2024-07-01)


{'BTCUSDT': 5, 'ETHUSDT': 5}

## Вспомогательные функции

- `make_backtest()` — единые параметры исполнения и защита от тихого “пропадания” издержек  
- `strategy_with_start_date()` — запрещает торговлю в warm-up сегменте  
- `optimize_on_train()` → `run_on_test()` → `extract_test_returns()`  
- `run_one_fold()` — “один фолд” как отдельная процедура


In [6]:
from src.wf_runner import run_one_fold

## Проверка на одном фолде (быстрый smoke-test)

In [7]:
import warnings

sym0 = "BTCUSDT"
cost0 = COST_SCENARIOS["Base"]
fold0 = folds_by_symbol[sym0][0]
spec0 = REGISTRY[0] if REGISTRY else None

if spec0 is None:
    raise RuntimeError("Registry is empty; nothing to test.")

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=RuntimeWarning)

    row0, ret0, hm0 = run_one_fold(
        sym=sym0,
        data=data_by_symbol[sym0],
        fold=fold0,
        spec=spec0,
        cost=cost0,
        bt_cfg=cfg.bt,
        obj_cfg=cfg.objective,
        save_heatmap=bool(cfg.artifacts.save_heatmaps and spec0.save_heatmap),
    )

print("OOS returns:", ret0.index.min().date(), "→", ret0.index.max().date(), "| n=", len(ret0))
pd.Series(row0).to_frame("value").head(14)

Backtest.optimize:   0%|          | 0/30 [00:00<?, ?it/s]

FractionalBacktest.run:   0%|          | 0/1036 [00:00<?, ?bar/s]

FractionalBacktest.run:   0%|          | 0/946 [00:00<?, ?bar/s]

FractionalBacktest.run:   0%|          | 0/846 [00:00<?, ?bar/s]

FractionalBacktest.run:   0%|          | 0/946 [00:00<?, ?bar/s]

...

FractionalBacktest.run:   0%|          | 0/796 [00:00<?, ?bar/s]

FractionalBacktest.run:   0%|          | 0/1036 [00:00<?, ?bar/s]

FractionalBacktest.run:   0%|          | 0/373 [00:00<?, ?bar/s]

OOS returns: 2023-01-01 → 2023-06-30 | n= 181


,value
symbol,BTCUSDT
cost,Base
fold_id,0
train_start,2020-01-01 00:00:00
train_end,2023-01-01 00:00:00
buffer_start,2022-04-24 00:00:00
test_start,2023-01-01 00:00:00
test_end,2023-07-01 00:00:00
strategy_code,T1
strategy_name,SMA_Crossover


## Полный прогон: (актив × cost) → все стратегии → stitched OOS + артефакты

In [8]:
from src.wf_experiment import run_full_experiment
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=RuntimeWarning)
    folds_best_all, returns_oos_all, bench_oos_all, heatmaps_all, bench_folds_all = run_full_experiment(
        cfg=cfg,
        data_by_symbol=data_by_symbol,
        folds_by_symbol=folds_by_symbol,
        registry=REGISTRY,
        bench=BENCH,
        report_mod=report_mod,   # нужно только если use_cache_if_exists=True
        logger=logger,
    )

folds_best_all.head(3)

2026-03-26 11:58:49,911 | INFO | RUN: BTCUSDT | cost=Low | folds=5 | strategies=12


FractionalBacktest.run:   0%|          | 0/432 [00:00<?, ?bar/s]

Backtest.optimize:   0%|          | 0/30 [00:00<?, ?it/s]

FractionalBacktest.run:   0%|          | 0/946 [00:00<?, ?bar/s]

FractionalBacktest.run:   0%|          | 0/846 [00:00<?, ?bar/s]

...

FractionalBacktest.run:   0%|          | 0/896 [00:00<?, ?bar/s]

FractionalBacktest.run:   0%|          | 0/896 [00:00<?, ?bar/s]

FractionalBacktest.run:   0%|          | 0/233 [00:00<?, ?bar/s]

2026-03-26 12:08:34,033 | INFO | DONE. folds_best=(360, 23) | returns_oos=(65664, 5) | bench=(5472, 5) | heatmaps=(1260, 9)


,symbol,cost,fold_id,train_start,train_end,buffer_start,test_start,test_end,strategy_code,strategy_name,...,oos_total_return,oos_cagr,oos_sharpe,oos_sortino,oos_maxdd,oos_calmar,oos_ann_vol,oos_n_bars,oos_trades,oos_exposure_pct
0,BTCUSDT,Low,0,2020-01-01,2023-01-01,2022-04-24,2023-01-01,2023-07-01,T1,SMA_Crossover,...,0.365438,0.874070,1.705109,3.346332,-0.188852,4.628324,0.419390,181,3,28.637413
1,BTCUSDT,Low,0,2020-01-01,2023-01-01,2022-04-24,2023-01-01,2023-07-01,T2,EMA_Crossover,...,0.216786,0.485390,1.177386,2.227542,-0.189461,2.561954,0.404813,181,3,30.254042
2,BTCUSDT,Low,0,2020-01-01,2023-01-01,2022-04-24,2023-01-01,2023-07-01,T3,Donchian_Breakout,...,0.268674,0.615894,1.464996,2.767648,-0.181600,3.391496,0.375168,181,5,26.327945


In [9]:
folds_best_all[folds_best_all['cost'] == 'Base']

,symbol,cost,fold_id,train_start,train_end,buffer_start,test_start,test_end,strategy_code,strategy_name,...,oos_total_return,oos_cagr,oos_sharpe,oos_sortino,oos_maxdd,oos_calmar,oos_ann_vol,oos_n_bars,oos_trades,oos_exposure_pct
60,BTCUSDT,Base,0,2020-01-01,2023-01-01,2022-04-24,2023-01-01,2023-07-01,T1,SMA_Crossover,...,0.362982,0.867280,1.696626,3.328913,-0.188852,4.592370,0.419329,181,3,28.637413
61,BTCUSDT,Base,0,2020-01-01,2023-01-01,2022-04-24,2023-01-01,2023-07-01,T2,EMA_Crossover,...,0.214597,0.480008,1.168641,2.210727,-0.189664,2.530839,0.404693,181,3,30.254042
62,BTCUSDT,Base,0,2020-01-01,2023-01-01,2022-04-24,2023-01-01,2023-07-01,T3,Donchian_Breakout,...,0.264873,0.606148,1.448764,2.735736,-0.182295,3.325093,0.375200,181,5,26.327945
63,BTCUSDT,Base,0,2020-01-01,2023-01-01,2022-04-24,2023-01-01,2023-07-01,M1,TSMOM,...,0.222852,0.500362,1.185365,2.252678,-0.189880,2.635148,0.413433,181,1,30.023095
64,BTCUSDT,Base,0,2020-01-01,2023-01-01,2022-04-24,2023-01-01,2023-07-01,R1,RSI_MR,...,0.077963,0.163451,1.428208,9.563150,-0.011574,14.122030,0.110074,181,1,0.923788
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,ETHUSDT,Base,4,2022-01-01,2025-01-01,2024-04-24,2025-01-01,2025-07-01,S1,MAFilter_RSI_MR,...,0.144894,0.313726,2.027962,4.986993,-0.044267,7.087182,0.139266,181,2,2.540416
296,ETHUSDT,Base,4,2022-01-01,2025-01-01,2024-04-24,2025-01-01,2025-07-01,S2,MA200Filter_Bollinger_MR,...,0.070187,0.146589,2.088379,98.138188,-0.001000,146.588569,0.066548,181,1,0.923788
297,ETHUSDT,Base,4,2022-01-01,2025-01-01,2024-04-24,2025-01-01,2025-07-01,S3,Breakout_Confirm_MA,...,-0.069666,-0.135513,-0.352413,-0.556742,-0.233265,-0.580940,0.292938,181,2,9.237875
298,ETHUSDT,Base,4,2022-01-01,2025-01-01,2024-04-24,2025-01-01,2025-07-01,S4,MA_Confirm_TSMOM,...,-0.032046,-0.063571,0.021228,0.032407,-0.212275,-0.299473,0.384556,181,3,15.935335


## Отчёты: лидерборды, stitched метрики, equity/drawdown, heatmaps

In [10]:
# === Reports: leaderboards, stitched metrics, benchmark comparisons ===
from src.artifacts import safe_write_parquet

# 1) Core summary tables
leaderboard = report_mod.leaderboard_by_strategy(folds_best_all)
stitched = report_mod.stitched_metrics(returns_oos_all)

safe_write_parquet(leaderboard, cfg.results_dir / "leaderboard.parquet")
safe_write_parquet(stitched, cfg.results_dir / "stitched_metrics.parquet")

display(leaderboard.head(10))
display(stitched.head(10))

# 2) NEW TABLE #1: stitched metrics for the benchmark (same format as stitched)
bench_stitched = report_mod.stitched_metrics(bench_oos_all)
safe_write_parquet(bench_stitched, cfg.results_dir / "bench_stitched_metrics.parquet")
display(bench_stitched.head(10))

# 3) Per-fold benchmark comparisons (win-rate + NEW TABLE #2: excess magnitude)
if not bench_folds_all.empty and not folds_best_all.empty:
    merged = folds_best_all.merge(
        bench_folds_all[["symbol", "cost", "fold_id", "bench_calmar", "bench_total_return"]],
        on=["symbol", "cost", "fold_id"],
        how="left",
        validate="many_to_one",
    )

    # win indicators
    merged["beat_bench_calmar"] = (merged["oos_calmar"] > merged["bench_calmar"]).astype(int)
    merged["beat_bench_total_return"] = (merged["oos_total_return"] > merged["bench_total_return"]).astype(int)

    # NEW: magnitude of outperformance (excess)
    merged["excess_calmar"] = merged["oos_calmar"] - merged["bench_calmar"]
    merged["excess_total_return"] = merged["oos_total_return"] - merged["bench_total_return"]

    # existing win-rate summary (beat)
    beat = (
        merged.groupby(["symbol", "cost", "strategy_id"])
        .agg(
            folds=("fold_id", "nunique"),
            beat_calmar_share=("beat_bench_calmar", "mean"),
            beat_return_share=("beat_bench_total_return", "mean"),
            calmar_median=("oos_calmar", "median"),
            calmar_worst=("oos_calmar", "min"),
            trades_median=("oos_trades", "median"),
            exposure_median=("oos_exposure_pct", "median"),
        )
        .reset_index()
        .sort_values(
            ["symbol", "cost", "beat_calmar_share", "calmar_median"],
            ascending=[True, True, False, False],
        )
    )

    # NEW TABLE #2: excess summary (how big is the edge, not just how often)
    excess_summary = (
        merged.groupby(["symbol", "cost", "strategy_id"])
        .agg(
            folds=("fold_id", "nunique"),
            excess_calmar_median=("excess_calmar", "median"),
            excess_calmar_mean=("excess_calmar", "mean"),
            excess_calmar_worst=("excess_calmar", "min"),
            excess_return_median=("excess_total_return", "median"),
            excess_return_mean=("excess_total_return", "mean"),
            excess_return_worst=("excess_total_return", "min"),
        )
        .reset_index()
        .sort_values(
            ["symbol", "cost", "excess_calmar_median", "excess_return_median"],
            ascending=[True, True, False, False],
        )
    )

    safe_write_parquet(beat, cfg.results_dir / "beat_benchmark_summary.parquet")
    safe_write_parquet(excess_summary, cfg.results_dir / "excess_vs_benchmark_summary.parquet")

    display(beat.head(15))
    display(excess_summary.head(15))

# 4) Generate figures (top-3 per symbol/cost + one heatmap example)
report_mod.generate_all_reports(cfg.out_dir, figures_dir=cfg.figures_dir)
logger.info("Figures saved to %s", cfg.figures_dir)

,symbol,cost,strategy_id,oos_total_return_median,oos_total_return_mean,oos_total_return_min,oos_total_return_max,oos_cagr_median,oos_cagr_mean,oos_cagr_min,...,oos_calmar_max,oos_trades_median,oos_trades_mean,oos_trades_min,oos_trades_max,oos_exposure_pct_median,oos_exposure_pct_mean,oos_exposure_pct_min,oos_exposure_pct_max,folds
11,BTCUSDT,Base,T3:Donchian_Breakout,0.375211,0.304002,0.050442,0.418691,0.894518,0.717087,0.104330,...,14.824586,2.0,2.6,1,5,21.100917,19.846123,11.778291,26.327945,5
6,BTCUSDT,Base,S3:Breakout_Confirm_MA,0.315938,0.290569,-0.017281,0.477951,0.739591,0.691629,-0.034542,...,14.824586,2.0,2.0,1,3,20.642202,20.718756,14.780600,28.406467,5
3,BTCUSDT,Base,R3:ZScore_MR,0.139248,0.124731,-0.037419,0.389274,0.298815,0.290125,-0.072862,...,74.590788,2.0,1.8,1,3,3.233256,3.961262,3.211009,6.466513,5
8,BTCUSDT,Base,S5:Ensemble_3Signals,0.325624,0.236709,-0.107031,0.377394,0.761801,0.559653,-0.204102,...,6.018306,2.0,2.0,1,3,30.963303,31.228485,18.577982,41.705069,5
0,BTCUSDT,Base,M1:TSMOM,0.253959,0.212017,-0.108987,0.377394,0.566621,0.496220,-0.207612,...,5.697348,2.0,3.0,1,6,30.023095,30.212637,20.642202,41.705069,5
1,BTCUSDT,Base,R1:RSI_MR,0.059734,0.098864,-0.037294,0.375516,0.121973,0.226345,-0.073391,...,14.122030,1.0,1.6,1,3,6.422018,6.718432,0.923788,13.394919,5
7,BTCUSDT,Base,S4:MA_Confirm_TSMOM,0.234768,0.215761,0.119068,0.269269,0.526419,0.480721,0.254650,...,4.777917,3.0,4.0,2,8,27.752294,26.386458,18.807339,33.640553,5
10,BTCUSDT,Base,T2:EMA_Crossover,0.230228,0.210857,0.082364,0.286563,0.508357,0.470124,0.173049,...,3.410226,2.0,2.4,2,3,29.816514,30.667333,25.688073,38.709677,5
2,BTCUSDT,Base,R2:Bollinger_MR,0.090432,0.162179,-0.037419,0.618063,0.190749,0.411387,-0.072862,...,15.532227,2.0,1.6,1,2,3.225806,8.892402,3.002309,23.325635,5
4,BTCUSDT,Base,S1:MAFilter_RSI_MR,0.082985,0.113091,-0.037419,0.295686,0.174408,0.256886,-0.072862,...,12.003057,3.0,2.4,1,4,4.147465,4.932084,2.752294,7.390300,5


,symbol,cost,strategy_id,n_bars,Total Return,CAGR,Ann. Vol,Sharpe,Sortino,MaxDD,Calmar,CDaR_95
11,BTCUSDT,Base,T3:Donchian_Breakout,912,2.657121,0.680271,0.339496,1.697101,3.110322,-0.182295,3.731701,0.164250
6,BTCUSDT,Base,S3:Breakout_Confirm_MA,912,2.412177,0.634292,0.346412,1.589786,2.895720,-0.187703,3.379228,0.187703
8,BTCUSDT,Base,S5:Ensemble_3Signals,912,1.727006,0.494067,0.416041,1.171478,2.083967,-0.275875,1.790908,0.275875
4,BTCUSDT,Base,S1:MAFilter_RSI_MR,912,0.655745,0.223617,0.189809,1.157115,2.012420,-0.132554,1.686991,0.046873
7,BTCUSDT,Base,S4:MA_Confirm_TSMOM,912,1.643201,0.475519,0.376777,1.219223,2.166182,-0.294921,1.612358,0.272876
0,BTCUSDT,Base,M1:TSMOM,912,1.474465,0.437073,0.406483,1.093779,1.926282,-0.291349,1.500171,0.276781
3,BTCUSDT,Base,R3:ZScore_MR,912,0.721948,0.242967,0.209388,1.141454,2.114237,-0.166269,1.461294,0.079725
2,BTCUSDT,Base,R2:Bollinger_MR,912,0.935970,0.302634,0.257394,1.153812,2.086683,-0.208514,1.451382,0.119917
10,BTCUSDT,Base,T2:EMA_Crossover,912,1.581256,0.461580,0.394870,1.157177,2.046673,-0.371277,1.243225,0.342544
5,BTCUSDT,Base,S2:MA200Filter_Bollinger_MR,912,0.688875,0.233357,0.202269,1.135753,2.198218,-0.198348,1.176505,0.098541


,symbol,cost,strategy_id,n_bars,Total Return,CAGR,Ann. Vol,Sharpe,Sortino,MaxDD,Calmar,CDaR_95
0,BTCUSDT,Base,BENCH:BuyHold,912,4.824628,1.024301,0.482162,1.702648,3.188455,-0.296117,3.459111,0.247711
1,BTCUSDT,High,BENCH:BuyHold,912,4.786898,1.019043,0.482159,1.697251,3.178646,-0.297031,3.430758,0.248688
2,BTCUSDT,Low,BENCH:BuyHold,912,4.842126,1.026733,0.482165,1.705137,3.192980,-0.295694,3.472277,0.247259
3,ETHUSDT,Base,BENCH:BuyHold,912,0.896786,0.292017,0.620347,0.719353,1.277108,-0.645606,0.452315,0.580847
4,ETHUSDT,High,BENCH:BuyHold,912,0.884499,0.288661,0.620337,0.715157,1.269767,-0.646527,0.446480,0.581936
5,ETHUSDT,Low,BENCH:BuyHold,912,0.902485,0.293569,0.620353,0.721290,1.280503,-0.645181,0.455019,0.580344


,symbol,cost,strategy_id,folds,beat_calmar_share,beat_return_share,calmar_median,calmar_worst,trades_median,exposure_median
11,BTCUSDT,Base,T3:Donchian_Breakout,5,0.8,0.2,5.870185,1.491188,2.0,21.100917
2,BTCUSDT,Base,R2:Bollinger_MR,5,0.6,0.0,2.358739,-0.762799,2.0,3.225806
6,BTCUSDT,Base,S3:Breakout_Confirm_MA,5,0.4,0.4,5.815117,-0.369931,2.0,20.642202
3,BTCUSDT,Base,R3:ZScore_MR,5,0.4,0.0,4.293863,-0.762799,2.0,3.233256
1,BTCUSDT,Base,R1:RSI_MR,5,0.4,0.0,2.931322,-1.552038,1.0,6.422018
4,BTCUSDT,Base,S1:MAFilter_RSI_MR,5,0.4,0.0,1.315753,-0.762799,3.0,4.147465
5,BTCUSDT,Base,S2:MA200Filter_Bollinger_MR,5,0.4,0.0,1.067064,-0.407075,3.0,6.697460
8,BTCUSDT,Base,S5:Ensemble_3Signals,5,0.2,0.0,4.053475,-0.784178,2.0,30.963303
0,BTCUSDT,Base,M1:TSMOM,5,0.2,0.0,3.169947,-0.751994,2.0,30.023095
7,BTCUSDT,Base,S4:MA_Confirm_TSMOM,5,0.2,0.0,2.588893,1.952928,3.0,27.752294


,symbol,cost,strategy_id,folds,excess_calmar_median,excess_calmar_mean,excess_calmar_worst,excess_return_median,excess_return_mean,excess_return_worst
11,BTCUSDT,Base,T3:Donchian_Breakout,5,1.395866,1.866587,-9.256718,-0.053298,-0.135075,-0.563380
2,BTCUSDT,Base,R2:Bollinger_MR,5,0.000737,-1.291446,-5.503179,-0.238147,-0.276897,-0.471415
6,BTCUSDT,Base,S3:Breakout_Confirm_MA,5,-0.514384,1.482695,-6.766694,-0.111948,-0.148507,-0.512316
1,BTCUSDT,Base,R1:RSI_MR,5,-0.904357,-0.068802,-6.026357,-0.315007,-0.340212,-0.750291
8,BTCUSDT,Base,S5:Ensemble_3Signals,5,-1.148157,-2.210422,-8.528337,-0.141568,-0.202368,-0.502630
3,BTCUSDT,Base,R3:ZScore_MR,5,-1.214781,10.868897,-5.162848,-0.323939,-0.314346,-0.438980
0,BTCUSDT,Base,M1:TSMOM,5,-1.230102,-2.568231,-9.946663,-0.157123,-0.227060,-0.605402
7,BTCUSDT,Base,S4:MA_Confirm_TSMOM,5,-1.811156,-2.681832,-9.557635,-0.170159,-0.223315,-0.577134
10,BTCUSDT,Base,T2:EMA_Crossover,5,-2.098419,-3.258168,-10.050972,-0.144513,-0.228219,-0.613656
4,BTCUSDT,Base,S1:MAFilter_RSI_MR,5,-3.144992,-1.173538,-5.303244,-0.412160,-0.325986,-0.532568


2026-03-26 12:08:48,265 | INFO | Figures saved to C:\Users\prosh\OneDrive\Рабочий стол\ДЗ МСК\4 course\Диплом\refactoring\figures


In [11]:
bench_stitched

,symbol,cost,strategy_id,n_bars,Total Return,CAGR,Ann. Vol,Sharpe,Sortino,MaxDD,Calmar,CDaR_95
0,BTCUSDT,Base,BENCH:BuyHold,912,4.824628,1.024301,0.482162,1.702648,3.188455,-0.296117,3.459111,0.247711
1,BTCUSDT,High,BENCH:BuyHold,912,4.786898,1.019043,0.482159,1.697251,3.178646,-0.297031,3.430758,0.248688
2,BTCUSDT,Low,BENCH:BuyHold,912,4.842126,1.026733,0.482165,1.705137,3.192980,-0.295694,3.472277,0.247259
3,ETHUSDT,Base,BENCH:BuyHold,912,0.896786,0.292017,0.620347,0.719353,1.277108,-0.645606,0.452315,0.580847
4,ETHUSDT,High,BENCH:BuyHold,912,0.884499,0.288661,0.620337,0.715157,1.269767,-0.646527,0.446480,0.581936
5,ETHUSDT,Low,BENCH:BuyHold,912,0.902485,0.293569,0.620353,0.721290,1.280503,-0.645181,0.455019,0.580344


In [12]:
stitched[(stitched['cost'] == 'Base') & (stitched['symbol'] == 'BTCUSDT')].sort_values(by='Total Return', ascending=False)

,symbol,cost,strategy_id,n_bars,Total Return,CAGR,Ann. Vol,Sharpe,Sortino,MaxDD,Calmar,CDaR_95
11,BTCUSDT,Base,T3:Donchian_Breakout,912,2.657121,0.680271,0.339496,1.697101,3.110322,-0.182295,3.731701,0.164250
6,BTCUSDT,Base,S3:Breakout_Confirm_MA,912,2.412177,0.634292,0.346412,1.589786,2.895720,-0.187703,3.379228,0.187703
8,BTCUSDT,Base,S5:Ensemble_3Signals,912,1.727006,0.494067,0.416041,1.171478,2.083967,-0.275875,1.790908,0.275875
7,BTCUSDT,Base,S4:MA_Confirm_TSMOM,912,1.643201,0.475519,0.376777,1.219223,2.166182,-0.294921,1.612358,0.272876
10,BTCUSDT,Base,T2:EMA_Crossover,912,1.581256,0.461580,0.394870,1.157177,2.046673,-0.371277,1.243225,0.342544
0,BTCUSDT,Base,M1:TSMOM,912,1.474465,0.437073,0.406483,1.093779,1.926282,-0.291349,1.500171,0.276781
9,BTCUSDT,Base,T1:SMA_Crossover,912,1.359102,0.409874,0.397941,1.060627,1.856020,-0.447175,0.916587,0.408707
2,BTCUSDT,Base,R2:Bollinger_MR,912,0.935970,0.302634,0.257394,1.153812,2.086683,-0.208514,1.451382,0.119917
3,BTCUSDT,Base,R3:ZScore_MR,912,0.721948,0.242967,0.209388,1.141454,2.114237,-0.166269,1.461294,0.079725
5,BTCUSDT,Base,S2:MA200Filter_Bollinger_MR,912,0.688875,0.233357,0.202269,1.135753,2.198218,-0.198348,1.176505,0.098541


In [13]:
stitched[(stitched['cost'] == 'Base') & (stitched['symbol'] == 'ETHUSDT')].sort_values(by='Total Return', ascending=False)

,symbol,cost,strategy_id,n_bars,Total Return,CAGR,Ann. Vol,Sharpe,Sortino,MaxDD,Calmar,CDaR_95
45,ETHUSDT,Base,T1:SMA_Crossover,912,1.052135,0.333370,0.429967,0.882524,1.484464,-0.442005,0.754223,0.345136
46,ETHUSDT,Base,T2:EMA_Crossover,912,0.847482,0.278470,0.450737,0.767421,1.329392,-0.305769,0.910720,0.301601
43,ETHUSDT,Base,S4:MA_Confirm_TSMOM,912,0.810537,0.268176,0.425669,0.769690,1.282343,-0.420858,0.637212,0.389707
38,ETHUSDT,Base,R2:Bollinger_MR,912,0.800335,0.265311,0.431847,0.758189,1.250645,-0.484881,0.547168,0.443708
39,ETHUSDT,Base,R3:ZScore_MR,912,0.772803,0.257531,0.358310,0.813285,1.440701,-0.342694,0.751489,0.224437
40,ETHUSDT,Base,S1:MAFilter_RSI_MR,912,0.621728,0.213493,0.194353,1.091769,1.889805,-0.143472,1.488045,0.092648
41,ETHUSDT,Base,S2:MA200Filter_Bollinger_MR,912,0.595848,0.205705,0.185132,1.102022,1.914349,-0.172695,1.191147,0.109729
42,ETHUSDT,Base,S3:Breakout_Confirm_MA,912,0.459243,0.163288,0.393993,0.579078,0.961992,-0.419534,0.389212,0.340712
44,ETHUSDT,Base,S5:Ensemble_3Signals,912,0.387361,0.140006,0.436703,0.515729,0.867565,-0.396181,0.353388,0.360759
36,ETHUSDT,Base,M1:TSMOM,912,0.238579,0.089406,0.457164,0.415755,0.663812,-0.493955,0.181001,0.486835


### Выбранные heatmap’ы (явно)

Для стратегий с **двумя параметрами** (например SMA crossover fast/slow, Donchian N/exit) имеет смысл построить heatmap поверхности train-objective на 1–2 фолдах.


In [14]:
# === Explicit heatmaps (if train_heatmaps were saved) ===
if heatmaps_all is not None and not heatmaps_all.empty:
    sym = "BTCUSDT"
    cost = "Base"
    fold_id = 0

    for strategy_id in ["T1:SMA_Crossover", "T3:Donchian_Breakout"]:
        out_path = cfg.figures_dir / f"heatmap__{sym}__{cost}__fold{fold_id}__{strategy_id.replace(':','_')}.png"

        report_mod.plot_heatmap(
            heatmaps_all,
            symbol=sym,
            cost=cost,
            fold_id=fold_id,
            strategy_id=strategy_id,
            out_path=out_path,
        )

        if out_path.exists():
            logger.info("Saved heatmap: %s", out_path)
        else:
            logger.warning("Heatmap skipped (degenerate/no data): %s", out_path)
else:
    logger.warning("heatmaps_all is empty; enable cfg.artifacts.save_heatmaps=True and rerun.")

2026-03-26 12:08:48,560 | INFO | Saved heatmap: C:\Users\prosh\OneDrive\Рабочий стол\ДЗ МСК\4 course\Диплом\refactoring\figures\heatmap__BTCUSDT__Base__fold0__T1_SMA_Crossover.png
2026-03-26 12:08:48,772 | INFO | Saved heatmap: C:\Users\prosh\OneDrive\Рабочий стол\ДЗ МСК\4 course\Диплом\refactoring\figures\heatmap__BTCUSDT__Base__fold0__T3_Donchian_Breakout.png


## RC/SPA: контроль множественных попыток (data-snooping) на stitched OOS

Universe: каждая стратегия как **процедура** (внутри фолда выбирает параметры на train, затем даёт stitched OOS).

Шаги:
- строим `d_{t,j} = r_{t,j} - r_{t,bench}`,
- запускаем Hansen SPA (если доступен `arch`) и White Reality Check (fallback).


In [15]:
stitched[["symbol","cost","strategy_id","MaxDD","CDaR_95","Calmar","CAGR"]].head(10)

,symbol,cost,strategy_id,MaxDD,CDaR_95,Calmar,CAGR
11,BTCUSDT,Base,T3:Donchian_Breakout,-0.182295,0.164250,3.731701,0.680271
6,BTCUSDT,Base,S3:Breakout_Confirm_MA,-0.187703,0.187703,3.379228,0.634292
8,BTCUSDT,Base,S5:Ensemble_3Signals,-0.275875,0.275875,1.790908,0.494067
4,BTCUSDT,Base,S1:MAFilter_RSI_MR,-0.132554,0.046873,1.686991,0.223617
7,BTCUSDT,Base,S4:MA_Confirm_TSMOM,-0.294921,0.272876,1.612358,0.475519
0,BTCUSDT,Base,M1:TSMOM,-0.291349,0.276781,1.500171,0.437073
3,BTCUSDT,Base,R3:ZScore_MR,-0.166269,0.079725,1.461294,0.242967
2,BTCUSDT,Base,R2:Bollinger_MR,-0.208514,0.119917,1.451382,0.302634
10,BTCUSDT,Base,T2:EMA_Crossover,-0.371277,0.342544,1.243225,0.461580
5,BTCUSDT,Base,S2:MA200Filter_Bollinger_MR,-0.198348,0.098541,1.176505,0.233357


In [16]:
from src.rc_spa import run_rc_spa_returns

# scientific display
pd.set_option("display.float_format", "{:.6f}".format)

rcspa_df = run_rc_spa_returns(
    symbols=list(data_by_symbol.keys()),
    costs=cfg.costs,
    returns_oos_long=returns_oos_all,
    bench_long=bench_oos_all,
    st_module=st,
    block_size=int(cfg.rc_spa.block_size),
    reps=int(cfg.rc_spa.reps),
    seed=int(cfg.rc_spa.seed),
    studentize=bool(cfg.rc_spa.studentize),
    alpha=0.05,
    logger=logger,
)

safe_write_parquet(rcspa_df, cfg.results_dir / "rc_spa_results.parquet")
rcspa_df

2026-03-26 12:08:48,834 | INFO | RC/SPA universe: BTCUSDT | Low | T=912 | models=12
2026-03-26 12:08:55,715 | INFO | RC/SPA universe: BTCUSDT | Base | T=912 | models=12
2026-03-26 12:08:58,302 | INFO | RC/SPA universe: BTCUSDT | High | T=912 | models=12
2026-03-26 12:09:00,917 | INFO | RC/SPA universe: ETHUSDT | Low | T=912 | models=12
2026-03-26 12:09:03,657 | INFO | RC/SPA universe: ETHUSDT | Base | T=912 | models=12
2026-03-26 12:09:06,371 | INFO | RC/SPA universe: ETHUSDT | High | T=912 | models=12


,symbol,cost,T,universe_size,rc_pvalue,rc_stat,rc_best_model,spa_pvalue_lower,spa_pvalue_consistent,spa_pvalue_upper,reps,block_size,studentize,spa_n_better_models
0,BTCUSDT,Low,912,12,0.998001,-0.000643,T3:Donchian_Breakout,0.650500,0.971000,0.995000,2000.000000,10.000000,1.000000,0.000000
1,BTCUSDT,Base,912,12,0.998001,-0.000671,T3:Donchian_Breakout,0.654000,0.975500,0.997000,2000.000000,10.000000,1.000000,0.000000
2,BTCUSDT,High,912,12,0.998001,-0.000682,T3:Donchian_Breakout,0.656500,0.975500,0.997000,2000.000000,10.000000,1.000000,0.000000
3,ETHUSDT,Low,912,12,0.899050,-0.000178,T1:SMA_Crossover,0.719500,0.890000,0.890000,2000.000000,10.000000,1.000000,0.000000
4,ETHUSDT,Base,912,12,0.902049,-0.000183,T1:SMA_Crossover,0.724000,0.891000,0.891000,2000.000000,10.000000,1.000000,0.000000
5,ETHUSDT,High,912,12,0.904048,-0.000194,T1:SMA_Crossover,0.724500,0.896500,0.896500,2000.000000,10.000000,1.000000,0.000000


In [17]:
print(rcspa_df)

    symbol  cost    T  universe_size  rc_pvalue   rc_stat  \
0  BTCUSDT   Low  912             12   0.998001 -0.000643   
1  BTCUSDT  Base  912             12   0.998001 -0.000671   
2  BTCUSDT  High  912             12   0.998001 -0.000682   
3  ETHUSDT   Low  912             12   0.899050 -0.000178   
4  ETHUSDT  Base  912             12   0.902049 -0.000183   
5  ETHUSDT  High  912             12   0.904048 -0.000194   

          rc_best_model  spa_pvalue_lower  spa_pvalue_consistent  \
0  T3:Donchian_Breakout          0.650500               0.971000   
1  T3:Donchian_Breakout          0.654000               0.975500   
2  T3:Donchian_Breakout          0.656500               0.975500   
3      T1:SMA_Crossover          0.719500               0.890000   
4      T1:SMA_Crossover          0.724000               0.891000   
5      T1:SMA_Crossover          0.724500               0.896500   

   spa_pvalue_upper        reps  block_size  studentize  spa_n_better_models  
0          0.995

In [18]:
# --- Risk-adjusted RC / SPA using a daily utility (penalized return) ---
from src.artifacts import safe_write_parquet
from src.rc_spa import run_rc_spa_utility

# scientific display
pd.set_option("display.float_format", "{:.4f}".format)

LAMBDA_LIST = [1.0, 2.0]
ALPHA = 0.05

rcspa_util_df = run_rc_spa_utility(
    symbols=list(data_by_symbol.keys()),
    costs=cfg.costs,
    returns_oos_long=returns_oos_all,
    bench_long=bench_oos_all,
    st_module=st,
    lambdas=LAMBDA_LIST,
    block_size=int(cfg.rc_spa.block_size),
    reps=int(cfg.rc_spa.reps),
    seed=int(cfg.rc_spa.seed),
    studentize=bool(cfg.rc_spa.studentize),
    alpha=float(ALPHA),
    logger=logger,
)

safe_write_parquet(rcspa_util_df, cfg.results_dir / "rc_spa_results_utility.parquet")
rcspa_util_df.sort_values(["symbol", "cost", "lam"])

2026-03-26 12:09:09,058 | INFO | Risk-RC/SPA universe: BTCUSDT | Low | lam=1.00 | T=912 | models=12
2026-03-26 12:09:11,675 | INFO | Risk-RC/SPA universe: BTCUSDT | Low | lam=2.00 | T=912 | models=12
2026-03-26 12:09:14,305 | INFO | Risk-RC/SPA universe: BTCUSDT | Base | lam=1.00 | T=912 | models=12
2026-03-26 12:09:16,891 | INFO | Risk-RC/SPA universe: BTCUSDT | Base | lam=2.00 | T=912 | models=12
2026-03-26 12:09:19,521 | INFO | Risk-RC/SPA universe: BTCUSDT | High | lam=1.00 | T=912 | models=12
2026-03-26 12:09:22,117 | INFO | Risk-RC/SPA universe: BTCUSDT | High | lam=2.00 | T=912 | models=12
2026-03-26 12:09:24,829 | INFO | Risk-RC/SPA universe: ETHUSDT | Low | lam=1.00 | T=912 | models=12
2026-03-26 12:09:27,610 | INFO | Risk-RC/SPA universe: ETHUSDT | Low | lam=2.00 | T=912 | models=12
2026-03-26 12:09:30,684 | INFO | Risk-RC/SPA universe: ETHUSDT | Base | lam=1.00 | T=912 | models=12
2026-03-26 12:09:33,324 | INFO | Risk-RC/SPA universe: ETHUSDT | Base | lam=2.00 | T=912 | mode

,symbol,cost,lam,T,universe_size,rc_pvalue,rc_stat,rc_best_model,spa_pvalue_lower,spa_pvalue_consistent,spa_pvalue_upper,reps,block_size,studentize,spa_n_better_models
2,BTCUSDT,Base,1.0000,912,12,0.0005,0.0046,R3:ZScore_MR,0.0000,0.0000,0.0000,2000.0000,10.0000,1.0000,7.0000
3,BTCUSDT,Base,2.0000,912,12,0.0005,0.0107,R3:ZScore_MR,0.0000,0.0000,0.0000,2000.0000,10.0000,1.0000,9.0000
4,BTCUSDT,High,1.0000,912,12,0.0005,0.0045,R3:ZScore_MR,0.0000,0.0000,0.0000,2000.0000,10.0000,1.0000,7.0000
5,BTCUSDT,High,2.0000,912,12,0.0005,0.0107,R3:ZScore_MR,0.0000,0.0000,0.0000,2000.0000,10.0000,1.0000,9.0000
0,BTCUSDT,Low,1.0000,912,12,0.0005,0.0046,R3:ZScore_MR,0.0000,0.0000,0.0000,2000.0000,10.0000,1.0000,7.0000
1,BTCUSDT,Low,2.0000,912,12,0.0005,0.0107,R3:ZScore_MR,0.0000,0.0000,0.0000,2000.0000,10.0000,1.0000,9.0000
8,ETHUSDT,Base,1.0000,912,12,0.0005,0.0065,S2:MA200Filter_Bollinger_MR,0.0000,0.0000,0.0000,2000.0000,10.0000,1.0000,9.0000
9,ETHUSDT,Base,2.0000,912,12,0.0005,0.0137,S2:MA200Filter_Bollinger_MR,0.0000,0.0000,0.0000,2000.0000,10.0000,1.0000,12.0000
10,ETHUSDT,High,1.0000,912,12,0.0005,0.0065,S2:MA200Filter_Bollinger_MR,0.0000,0.0000,0.0000,2000.0000,10.0000,1.0000,9.0000
11,ETHUSDT,High,2.0000,912,12,0.0005,0.0136,S2:MA200Filter_Bollinger_MR,0.0000,0.0000,0.0000,2000.0000,10.0000,1.0000,12.0000


In [19]:
print(rcspa_util_df.sort_values(["symbol", "cost", "lam"]))

     symbol  cost    lam    T  universe_size  rc_pvalue  rc_stat  \
2   BTCUSDT  Base 1.0000  912             12     0.0005   0.0046   
3   BTCUSDT  Base 2.0000  912             12     0.0005   0.0107   
4   BTCUSDT  High 1.0000  912             12     0.0005   0.0045   
5   BTCUSDT  High 2.0000  912             12     0.0005   0.0107   
0   BTCUSDT   Low 1.0000  912             12     0.0005   0.0046   
1   BTCUSDT   Low 2.0000  912             12     0.0005   0.0107   
8   ETHUSDT  Base 1.0000  912             12     0.0005   0.0065   
9   ETHUSDT  Base 2.0000  912             12     0.0005   0.0137   
10  ETHUSDT  High 1.0000  912             12     0.0005   0.0065   
11  ETHUSDT  High 2.0000  912             12     0.0005   0.0136   
6   ETHUSDT   Low 1.0000  912             12     0.0005   0.0065   
7   ETHUSDT   Low 2.0000  912             12     0.0005   0.0137   

                  rc_best_model  spa_pvalue_lower  spa_pvalue_consistent  \
2                  R3:ZScore_MR        

In [20]:
from src.artifacts import safe_write_parquet
from src.metrics import compute_metrics_from_returns
from src.rc_spa import run_spa_better_strategies_table

pd.set_option("display.float_format", "{:.4f}".format)

LAM_STAR = 1.0
ALPHA = 0.05
PV_TYPE = "consistent"
MAX_SHOW = 20

tbl, tbl_view = run_spa_better_strategies_table(
    symbols=list(data_by_symbol.keys()),
    costs=cfg.costs,
    returns_oos_long=returns_oos_all,
    bench_long=bench_oos_all,
    compute_metrics_fn=compute_metrics_from_returns,
    block_size=int(cfg.rc_spa.block_size),
    reps=int(cfg.rc_spa.reps),
    seed=int(cfg.rc_spa.seed),
    studentize=bool(cfg.rc_spa.studentize),
    lam_star=float(LAM_STAR),
    alpha=float(ALPHA),
    pvalue_type=PV_TYPE,
    max_show=int(MAX_SHOW),
    logger=logger,
)

safe_write_parquet(tbl, cfg.results_dir / f"spa_better_strategies_table_lam{LAM_STAR}.parquet")
display(tbl_view)

2026-03-26 12:09:41,720 | INFO | SPA better models: BTCUSDT | Low | lam=1.00 | alpha=0.050 | n=7
2026-03-26 12:09:42,239 | INFO | SPA better models: BTCUSDT | Base | lam=1.00 | alpha=0.050 | n=7
2026-03-26 12:09:42,748 | INFO | SPA better models: BTCUSDT | High | lam=1.00 | alpha=0.050 | n=7
2026-03-26 12:09:43,288 | INFO | SPA better models: ETHUSDT | Low | lam=1.00 | alpha=0.050 | n=9
2026-03-26 12:09:43,938 | INFO | SPA better models: ETHUSDT | Base | lam=1.00 | alpha=0.050 | n=9
2026-03-26 12:09:44,497 | INFO | SPA better models: ETHUSDT | High | lam=1.00 | alpha=0.050 | n=9


,symbol,cost,lam,strategy_id,utility_mean,utility_mean_excess,TotalReturn,CAGR,VolAnn,Sharpe,Sortino,MaxDD,Calmar,n_bars
8,BTCUSDT,Base,1.0000,BENCH:BuyHold,-0.0047,0.0000,4.8246,1.0243,0.4822,1.7026,3.1885,-0.2961,3.4591,912
11,BTCUSDT,Base,1.0000,R3:ZScore_MR,-0.0001,0.0046,0.7219,0.2430,0.2094,1.1415,2.1142,-0.1663,1.4613,912
13,BTCUSDT,Base,1.0000,S2:MA200Filter_Bollinger_MR,-0.0002,0.0045,0.6889,0.2334,0.2023,1.1358,2.1982,-0.1983,1.1765,912
12,BTCUSDT,Base,1.0000,S1:MAFilter_RSI_MR,-0.0002,0.0045,0.6557,0.2236,0.1898,1.1571,2.0124,-0.1326,1.6870,912
9,BTCUSDT,Base,1.0000,R1:RSI_MR,-0.0008,0.0039,0.5406,0.1888,0.2284,0.8702,1.4686,-0.1904,0.9917,912
10,BTCUSDT,Base,1.0000,R2:Bollinger_MR,-0.0008,0.0038,0.9360,0.3026,0.2574,1.1538,2.0867,-0.2085,1.4514,912
15,BTCUSDT,Base,1.0000,T3:Donchian_Breakout,-0.0017,0.0030,2.6571,0.6803,0.3395,1.6971,3.1103,-0.1823,3.7317,912
14,BTCUSDT,Base,1.0000,S3:Breakout_Confirm_MA,-0.0020,0.0027,2.4122,0.6343,0.3464,1.5898,2.8957,-0.1877,3.3792,912
16,BTCUSDT,High,1.0000,BENCH:BuyHold,-0.0047,0.0000,4.7869,1.0190,0.4822,1.6973,3.1786,-0.2970,3.4308,912
19,BTCUSDT,High,1.0000,R3:ZScore_MR,-0.0001,0.0045,0.7019,0.2372,0.2095,1.1186,2.0633,-0.1663,1.4264,912


## Robustness: издержки и подпериоды (на stitched returns)

In [21]:
from src import robustness as rb

symbols = list(data_by_symbol.keys())

SUBPERIODS = {
    "2023": ("2023-01-01", "2024-01-01"),
    "2024": ("2024-01-01", "2025-01-01"),
    "2025": ("2025-01-01", "2025-07-01"),
}

# 1) Cost sensitivity: top by Calmar (Base) across Low/Base/High + BENCH
sensitivity_calmar_df = rb.cost_sensitivity_top_calmar(
    cfg=cfg,
    stitched=stitched,
    returns_oos_all=returns_oos_all,
    bench_oos_all=bench_oos_all,
    symbols=symbols,
    topN=5,
)

# 2) Cost sensitivity: top by UtilityExcess (worsen-DD), lam_star=1.0
sensitivity_utility_df = rb.cost_sensitivity_top_utility(
    cfg=cfg,
    stitched=stitched,
    returns_oos_all=returns_oos_all,
    bench_oos_all=bench_oos_all,
    symbols=symbols,
    lam_star=1.0,
    topN=5,
)

# 3) Subperiods: top-3 by Calmar (Base) + BENCH
subperiod_calmar_df = rb.subperiods_top_calmar(
    cfg=cfg,
    stitched=stitched,
    returns_oos_all=returns_oos_all,
    bench_oos_all=bench_oos_all,
    symbols=symbols,
    subperiods=SUBPERIODS,
    topK=3,
)

# 4) Subperiods: top-3 by UtilityExcess (Base) + BENCH
subperiod_utility_df = rb.subperiods_top_utility(
    cfg=cfg,
    stitched=stitched,
    returns_oos_all=returns_oos_all,
    bench_oos_all=bench_oos_all,
    symbols=symbols,
    subperiods=SUBPERIODS,
    lam_star=1.0,
    topK=3,
)

display(sensitivity_calmar_df.head(15))
display(sensitivity_utility_df.head(15))
display(subperiod_calmar_df.head(15))
display(subperiod_utility_df.head(15))

,symbol,cost,strategy_id,n_bars,Total Return,CAGR,Ann. Vol,Sharpe,Sortino,MaxDD,Calmar,CDaR_95,selection_rule
0,BTCUSDT,Base,BENCH:BuyHold,912,4.8246,1.0243,0.4822,1.7026,3.1885,-0.2961,3.4591,0.2477,Top5_by_Calmar_on_Base
1,BTCUSDT,High,BENCH:BuyHold,912,4.7869,1.0190,0.4822,1.6973,3.1786,-0.2970,3.4308,0.2487,Top5_by_Calmar_on_Base
2,BTCUSDT,Low,BENCH:BuyHold,912,4.8421,1.0267,0.4822,1.7051,3.1930,-0.2957,3.4723,0.2473,Top5_by_Calmar_on_Base
3,BTCUSDT,Base,S1:MAFilter_RSI_MR,912,0.6557,0.2236,0.1898,1.1571,2.0124,-0.1326,1.6870,0.0469,Top5_by_Calmar_on_Base
4,BTCUSDT,High,S1:MAFilter_RSI_MR,912,0.6301,0.2160,0.1900,1.1234,1.9423,-0.1326,1.6295,0.0484,Top5_by_Calmar_on_Base
5,BTCUSDT,Low,S1:MAFilter_RSI_MR,912,0.6677,0.2271,0.1897,1.1727,2.0447,-0.1326,1.7136,0.0462,Top5_by_Calmar_on_Base
6,BTCUSDT,Base,S3:Breakout_Confirm_MA,912,2.4122,0.6343,0.3464,1.5898,2.8957,-0.1877,3.3792,0.1877,Top5_by_Calmar_on_Base
7,BTCUSDT,High,S3:Breakout_Confirm_MA,912,2.3681,0.6258,0.3463,1.5750,2.8674,-0.1892,3.3083,0.1892,Top5_by_Calmar_on_Base
8,BTCUSDT,Low,S3:Breakout_Confirm_MA,912,2.4327,0.6382,0.3464,1.5966,2.9087,-0.1870,3.4127,0.1870,Top5_by_Calmar_on_Base
9,BTCUSDT,Base,S4:MA_Confirm_TSMOM,912,1.6432,0.4755,0.3768,1.2192,2.1662,-0.2949,1.6124,0.2729,Top5_by_Calmar_on_Base


,symbol,cost,strategy_id,n_bars,Total Return,CAGR,Ann. Vol,Sharpe,Sortino,MaxDD,Calmar,CDaR_95,selection_rule,lam
0,BTCUSDT,Base,BENCH:BuyHold,912,4.8246,1.0243,0.4822,1.7026,3.1885,-0.2961,3.4591,0.2477,Top5_by_UtilityExcessWorsenDD_lam1.0_on_Base,1.0000
1,BTCUSDT,High,BENCH:BuyHold,912,4.7869,1.0190,0.4822,1.6973,3.1786,-0.2970,3.4308,0.2487,Top5_by_UtilityExcessWorsenDD_lam1.0_on_Base,1.0000
2,BTCUSDT,Low,BENCH:BuyHold,912,4.8421,1.0267,0.4822,1.7051,3.1930,-0.2957,3.4723,0.2473,Top5_by_UtilityExcessWorsenDD_lam1.0_on_Base,1.0000
3,BTCUSDT,Base,R1:RSI_MR,912,0.5406,0.1888,0.2284,0.8702,1.4686,-0.1904,0.9917,0.1146,Top5_by_UtilityExcessWorsenDD_lam1.0_on_Base,1.0000
4,BTCUSDT,High,R1:RSI_MR,912,0.5246,0.1839,0.2287,0.8510,1.4327,-0.1904,0.9658,0.1149,Top5_by_UtilityExcessWorsenDD_lam1.0_on_Base,1.0000
5,BTCUSDT,Low,R1:RSI_MR,912,0.5480,0.1911,0.2282,0.8790,1.4850,-0.1904,1.0037,0.1145,Top5_by_UtilityExcessWorsenDD_lam1.0_on_Base,1.0000
6,BTCUSDT,Base,R2:Bollinger_MR,912,0.9360,0.3026,0.2574,1.1538,2.0867,-0.2085,1.4514,0.1199,Top5_by_UtilityExcessWorsenDD_lam1.0_on_Base,1.0000
7,BTCUSDT,High,R2:Bollinger_MR,912,0.9159,0.2972,0.2576,1.1370,2.0526,-0.2085,1.4254,0.1204,Top5_by_UtilityExcessWorsenDD_lam1.0_on_Base,1.0000
8,BTCUSDT,Low,R2:Bollinger_MR,912,0.9453,0.3051,0.2573,1.1615,2.1023,-0.2085,1.4634,0.1197,Top5_by_UtilityExcessWorsenDD_lam1.0_on_Base,1.0000
9,BTCUSDT,Base,R3:ZScore_MR,912,0.7219,0.2430,0.2094,1.1415,2.1142,-0.1663,1.4613,0.0797,Top5_by_UtilityExcessWorsenDD_lam1.0_on_Base,1.0000


,symbol,cost,strategy_id,subperiod,n_bars,Total Return,CAGR,Ann. Vol,Sharpe,Sortino,MaxDD,Calmar,CDaR_95
0,BTCUSDT,Base,BENCH:BuyHold,2023,365,1.5134,1.5134,0.4401,2.3137,4.6192,-0.2000,7.5658,0.1786
1,BTCUSDT,Base,BENCH:BuyHold,2024,366,1.0275,1.0236,0.5277,1.5984,2.9810,-0.2856,3.5846,0.2544
2,BTCUSDT,Base,BENCH:BuyHold,2025,181,0.1430,0.3094,0.4659,0.8107,1.4092,-0.2810,1.1011,0.2450
3,BTCUSDT,Base,T3:Donchian_Breakout,2023,365,0.7845,0.7845,0.3430,1.8588,3.5488,-0.1823,4.3033,0.1804
4,BTCUSDT,Base,T3:Donchian_Breakout,2024,366,0.9510,0.9474,0.3942,1.8869,3.4002,-0.1524,6.2175,0.1278
5,BTCUSDT,Base,T3:Donchian_Breakout,2025,181,0.0504,0.1043,0.1689,0.6711,1.1741,-0.0700,1.4912,0.0700
6,BTCUSDT,Base,S3:Breakout_Confirm_MA,2023,365,0.8565,0.8565,0.3541,1.9225,3.7865,-0.1499,5.7123,0.1436
7,BTCUSDT,Base,S3:Breakout_Confirm_MA,2024,366,0.8703,0.8671,0.3931,1.7840,3.1363,-0.1870,4.6356,0.1358
8,BTCUSDT,Base,S3:Breakout_Confirm_MA,2025,181,-0.0173,-0.0345,0.1953,-0.0831,-0.1328,-0.0934,-0.3699,0.0934
9,BTCUSDT,Base,S5:Ensemble_3Signals,2023,365,0.6665,0.6665,0.4034,1.4658,2.7551,-0.1889,3.5291,0.1714


,symbol,cost,strategy_id,subperiod,lam,n_bars,Total Return,CAGR,Ann. Vol,Sharpe,Sortino,MaxDD,Calmar,CDaR_95
0,BTCUSDT,Base,BENCH:BuyHold,2023,1.0000,365,1.5134,1.5134,0.4401,2.3137,4.6192,-0.2000,7.5658,0.1786
1,BTCUSDT,Base,BENCH:BuyHold,2024,1.0000,366,1.0275,1.0236,0.5277,1.5984,2.9810,-0.2856,3.5846,0.2544
2,BTCUSDT,Base,BENCH:BuyHold,2025,1.0000,181,0.1430,0.3094,0.4659,0.8107,1.4092,-0.2810,1.1011,0.2450
3,BTCUSDT,Base,R3:ZScore_MR,2023,1.0000,365,0.3373,0.3373,0.1953,1.5828,3.8938,-0.0964,3.4979,0.0430
4,BTCUSDT,Base,R3:ZScore_MR,2024,1.0000,366,0.3079,0.3070,0.1827,1.5542,3.2785,-0.0743,4.1313,0.0098
5,BTCUSDT,Base,R3:ZScore_MR,2025,1.0000,181,-0.0155,-0.0310,0.2762,0.0240,0.0351,-0.1663,-0.1865,0.1257
6,BTCUSDT,Base,S2:MA200Filter_Bollinger_MR,2023,1.0000,365,0.4513,0.4513,0.2016,1.9462,4.8067,-0.0964,4.6801,0.0430
7,BTCUSDT,Base,S2:MA200Filter_Bollinger_MR,2024,1.0000,366,0.0416,0.0414,0.2091,0.2968,0.4882,-0.1983,0.2090,0.1268
8,BTCUSDT,Base,S2:MA200Filter_Bollinger_MR,2025,1.0000,181,0.1173,0.2506,0.1885,1.2784,2.5870,-0.0818,3.0642,0.0433
9,BTCUSDT,Base,S1:MAFilter_RSI_MR,2023,1.0000,365,0.2472,0.2472,0.1964,1.2216,2.2078,-0.0964,2.5637,0.0435


In [22]:
display(subperiod_calmar_df)


,symbol,cost,strategy_id,subperiod,n_bars,Total Return,CAGR,Ann. Vol,Sharpe,Sortino,MaxDD,Calmar,CDaR_95
0,BTCUSDT,Base,BENCH:BuyHold,2023,365,1.5134,1.5134,0.4401,2.3137,4.6192,-0.2000,7.5658,0.1786
1,BTCUSDT,Base,BENCH:BuyHold,2024,366,1.0275,1.0236,0.5277,1.5984,2.9810,-0.2856,3.5846,0.2544
2,BTCUSDT,Base,BENCH:BuyHold,2025,181,0.1430,0.3094,0.4659,0.8107,1.4092,-0.2810,1.1011,0.2450
3,BTCUSDT,Base,T3:Donchian_Breakout,2023,365,0.7845,0.7845,0.3430,1.8588,3.5488,-0.1823,4.3033,0.1804
4,BTCUSDT,Base,T3:Donchian_Breakout,2024,366,0.9510,0.9474,0.3942,1.8869,3.4002,-0.1524,6.2175,0.1278
5,BTCUSDT,Base,T3:Donchian_Breakout,2025,181,0.0504,0.1043,0.1689,0.6711,1.1741,-0.0700,1.4912,0.0700
6,BTCUSDT,Base,S3:Breakout_Confirm_MA,2023,365,0.8565,0.8565,0.3541,1.9225,3.7865,-0.1499,5.7123,0.1436
7,BTCUSDT,Base,S3:Breakout_Confirm_MA,2024,366,0.8703,0.8671,0.3931,1.7840,3.1363,-0.1870,4.6356,0.1358
8,BTCUSDT,Base,S3:Breakout_Confirm_MA,2025,181,-0.0173,-0.0345,0.1953,-0.0831,-0.1328,-0.0934,-0.3699,0.0934
9,BTCUSDT,Base,S5:Ensemble_3Signals,2023,365,0.6665,0.6665,0.4034,1.4658,2.7551,-0.1889,3.5291,0.1714


---

## Примечания по методологии (для текста курсовой)

1) **Warm-up без утечек:** тест запускается на (buffer+test), но торговля до `test_start` запрещена через `start_date`.

2) **Universe в RC/SPA:** здесь universe = набор “процедур” (по одной stitched OOS серии на стратегию), где параметры выбираются внутри каждого фолда только по train.

3) Если захочешь **строже** (universe = стратегия×параметры), тогда нужно сохранять OOS returns для каждой параметр-комбинации или хотя бы top-K кандидатов в каждом фолде — и уже на этой матрице запускать RC/SPA.

4) **Скорость:** полный прогон (BTC+ETH × 3 cost × все стратегии × все фолды × grid) может быть тяжёлым. Для отладки включи `CFG["debug"]["enabled"]=True` и/или запускай только Base costs.


In [23]:
stitched[
    (stitched['cost'] == 'Base')
][['symbol', 'strategy_id', 'Total Return', 'CAGR']].sort_values(
    ['symbol', 'Total Return'], ascending=[True, False]
)

,symbol,strategy_id,Total Return,CAGR
11,BTCUSDT,T3:Donchian_Breakout,2.6571,0.6803
6,BTCUSDT,S3:Breakout_Confirm_MA,2.4122,0.6343
8,BTCUSDT,S5:Ensemble_3Signals,1.7270,0.4941
7,BTCUSDT,S4:MA_Confirm_TSMOM,1.6432,0.4755
10,BTCUSDT,T2:EMA_Crossover,1.5813,0.4616
0,BTCUSDT,M1:TSMOM,1.4745,0.4371
9,BTCUSDT,T1:SMA_Crossover,1.3591,0.4099
2,BTCUSDT,R2:Bollinger_MR,0.9360,0.3026
3,BTCUSDT,R3:ZScore_MR,0.7219,0.2430
5,BTCUSDT,S2:MA200Filter_Bollinger_MR,0.6889,0.2334


In [24]:
bench_stitched[
    bench_stitched['cost'] == 'Base'
][['symbol', 'strategy_id', 'Total Return', 'CAGR']]

,symbol,strategy_id,Total Return,CAGR
0,BTCUSDT,BENCH:BuyHold,4.8246,1.0243
3,ETHUSDT,BENCH:BuyHold,0.8968,0.2920
